# Test del servidor MCP de censoargentino

Este notebook prueba el servidor MCP de dos formas:

1. **Test directo** — llama las funciones del server como Python puro. Rápido, sin overhead de protocolo.
2. **Test MCP completo** — usa el cliente MCP para hacer el round-trip real (stdio → JSON-RPC → respuesta). Es exactamente lo que hace Claude Desktop o Cursor cuando usa el server.

**Prerequisito:** `pip install "censoargentino[mcp]"`

In [2]:
import sys

# Instalar / actualizar censoargentino con soporte MCP
!{sys.executable} -m pip install -q --upgrade "censoargentino[mcp]"

# Verificar versiones
import importlib.metadata
for pkg in ["mcp", "censoargentino"]:
    try:
        v = importlib.metadata.version(pkg)
        print(f"✓ {pkg} {v}")
    except Exception:
        print(f"✗ {pkg} — instalación fallida")

✓ mcp 1.26.0
✓ censoargentino 0.1.6



[notice] A new release of pip is available: 25.0.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


> ⚠️ **Reiniciá el kernel** antes de continuar (`Kernel → Restart Kernel`) para que los paquetes recién instalados queden disponibles.

In [3]:
# Imports generales
import json
import asyncio
import subprocess
import sys
from io import StringIO

import pandas as pd

# Helper: pd.read_json necesita StringIO en pandas recientes (evita confusión con filepath)
def _df(json_str, **kw):
    return pd.read_json(StringIO(json_str), **kw)

print("Imports OK")

Imports OK


---
## Parte 1 — Test directo (funciones Python)

Importa las funciones del server y las llama directamente. Útil para verificar la lógica sin levantar el proceso MCP.

In [4]:
from censoargentino.mcp_server import (
    buscar_variables,
    describir_variable,
    tabla,
    comparar,
    consultar,
    resource_provincias,
    resource_variables,
)

print("Funciones del server importadas correctamente")

Funciones del server importadas correctamente


In [5]:
# ── Resource: provincias ──────────────────────────────────
print("=== censo://provincias ===")
provincias = json.loads(resource_provincias())
print(f"{len(provincias)} provincias")
for p in provincias[:5]:
    print(f"  {p['codigo']}  {p['provincia']}")
print("  ...")

=== censo://provincias ===
24 provincias
  02  Ciudad Autónoma De Buenos Aires
  06  Buenos Aires
  10  Catamarca
  14  Córdoba
  18  Corrientes
  ...


In [6]:
# ── Tool: buscar_variables ────────────────────────────────
print("=== buscar_variables('nbi') ===")
resultado = buscar_variables("nbi")
for v in json.loads(resultado):
    print(f"  {v['codigo_variable']:30s}  {v['etiqueta_variable']}")

[censo] Descargando catalogo de variables (archivo de metadatos, ~1 MB)...
[censo] Iniciando DuckDB e instalando extension HTTP...


=== buscar_variables('nbi') ===


[censo] Listo. Las consultas van directo a Hugging Face (pedroorden/censoargentino).
[censo] Catalogo cargado en 1.3s -> 86 variables disponibles
[censo]   Entidades: PERSONA (personas), HOGAR (hogares), VIVIENDA (viviendas)


  HOGAR_NBI_ESC                   NBI Escolaridad
  HOGAR_NBI_HAC                   NBI Hacinamiento
  HOGAR_NBI_SAN                   NBI Condiciones sanitarias
  HOGAR_NBI_SUB                   NBI Capacidad de subsistencia
  HOGAR_NBI_TOT                   Necesidades básicas insatisfechas
  HOGAR_NBI_VIV                   NBI Vivienda de tipo inconveniente


In [7]:
# ── Tool: buscar_variables con entidad ───────────────────
print("=== buscar_variables('internet', entidad='HOGAR') ===")
resultado = buscar_variables("internet", entidad="HOGAR")
for v in json.loads(resultado):
    print(f"  {v['codigo_variable']:30s}  {v['etiqueta_variable']}")

=== buscar_variables('internet', entidad='HOGAR') ===
  HOGAR_H24A                      El hogar tiene internet en la vivienda
  HOGAR_H24B                      El hogar tiene celular con internet


In [8]:
# ── Tool: describir_variable ──────────────────────────────
print("=== describir_variable('HOGAR_NBI_TOT') ===")
detalle = json.loads(describir_variable("HOGAR_NBI_TOT"))
print(f"  Código     : {detalle['codigo']}")
print(f"  Descripción: {detalle['descripcion']}")
print(f"  Entidad    : {detalle['entidad']}")
print(f"  Categorías :")
for cat in detalle['categorias']:
    print(f"    {cat['valor_categoria']}  →  {cat['etiqueta_categoria']}")

=== describir_variable('HOGAR_NBI_TOT') ===
  Código     : HOGAR_NBI_TOT
  Descripción: Necesidades básicas insatisfechas
  Entidad    : HOGAR
  Categorías :
    1  →  Sí
    2  →  No


In [9]:
# ── Tool: tabla ───────────────────────────────────────────
print("=== tabla('HOGAR_NBI_TOT', provincia='Chaco') ===")
resultado = tabla("HOGAR_NBI_TOT", provincia="Chaco")
display(_df(resultado))

[censo] =======================================================
[censo] Consulta al Censo Nacional 2022 (INDEC)
[censo] Fuente: censo-2022-largo.parquet (~137 MB en Hugging Face)
[censo]   (DuckDB descarga solo los bloques que coinciden con los filtros)
[censo] -------------------------------------------------------
[censo]   Variable : HOGAR_NBI_TOT  ("Necesidades básicas insatisfechas")
[censo]   Provincia: Chaco  (codigo INDEC: 22)
[censo] -------------------------------------------------------
[censo] Estructura del resultado:
[censo]   Cada fila = una (radio censal x categoria de variable x conteo)
[censo]   Columnas clave: id_geo | codigo_variable | valor_categoria
[censo]                   etiqueta_categoria | conteo
[censo] =======================================================
[censo] Descargando datos desde Hugging Face...


=== tabla('HOGAR_NBI_TOT', provincia='Chaco') ===


[censo] Descarga completa en 3.2s -> 3,803 filas | 13 columnas | 2.3 MB en memoria


,categoria,N,%
0,Sí,42014,11.2
1,No,332473,88.8


In [10]:
# ── Tool: tabla nacional ──────────────────────────────────
print("=== tabla('PERSONA_P02') — escala nacional ===")
resultado = tabla("PERSONA_P02")
display(_df(resultado))

[censo] =======================================================
[censo] Consulta al Censo Nacional 2022 (INDEC)
[censo] Fuente: censo-2022-largo.parquet (~137 MB en Hugging Face)
[censo]   (DuckDB descarga solo los bloques que coinciden con los filtros)
[censo] -------------------------------------------------------
[censo]   Variable : PERSONA_P02  ("Sexo registrado al nacer")
[censo] -------------------------------------------------------
[censo] Estructura del resultado:
[censo]   Cada fila = una (radio censal x categoria de variable x conteo)
[censo]   Columnas clave: id_geo | codigo_variable | valor_categoria
[censo]                   etiqueta_categoria | conteo
[censo] =======================================================
[censo] Descargando datos desde Hugging Face...


=== tabla('PERSONA_P02') — escala nacional ===


[censo] Descarga completa en 3.0s -> 132,571 filas | 13 columnas | 84.2 MB en memoria


,categoria,N,%
0,Mujer / Femenino,23607906,51.8
1,Varón / Masculino,22010881,48.2


In [11]:
# ── Tool: comparar provincias ─────────────────────────────
print("=== comparar('HOGAR_NBI_TOT', nivel='provincia') ===")
resultado = comparar("HOGAR_NBI_TOT", nivel="provincia")
display(_df(resultado, orient="index").head(8))

[censo] =======================================================
[censo] Consulta al Censo Nacional 2022 (INDEC)
[censo] Fuente: censo-2022-largo.parquet (~137 MB en Hugging Face)
[censo]   (DuckDB descarga solo los bloques que coinciden con los filtros)
[censo] -------------------------------------------------------
[censo]   Variable : HOGAR_NBI_TOT  ("Necesidades básicas insatisfechas")
[censo] -------------------------------------------------------
[censo] Estructura del resultado:
[censo]   Cada fila = una (radio censal x categoria de variable x conteo)
[censo]   Columnas clave: id_geo | codigo_variable | valor_categoria
[censo]                   etiqueta_categoria | conteo
[censo] =======================================================
[censo] Descargando datos desde Hugging Face...


=== comparar('HOGAR_NBI_TOT', nivel='provincia') ===


[censo] Descarga completa en 0.3s -> 127,347 filas | 13 columnas | 79.3 MB en memoria


,No,Sí,Total
Buenos Aires,93.7,6.3,6051550
Caba,95.2,4.8,1406735
Córdoba,95.7,4.3,1394400
Santa Fe,95.1,4.9,1289967
Mendoza,93.6,6.4,652184
Tucumán,90.5,9.5,505542
Entre Ríos,94.0,6.0,500660
Misiones,91.3,8.7,425667


In [12]:
# ── Tool: comparar subset de provincias ───────────────────
print("=== comparar — solo NEA ===")
resultado = comparar(
    "HOGAR_NBI_TOT",
    nivel="provincia",
    provincias=["Chaco", "Formosa", "Misiones", "Corrientes"]
)
display(_df(resultado, orient="index"))

[censo] =======================================================
[censo] Consulta al Censo Nacional 2022 (INDEC)
[censo] Fuente: censo-2022-largo.parquet (~137 MB en Hugging Face)
[censo]   (DuckDB descarga solo los bloques que coinciden con los filtros)
[censo] -------------------------------------------------------
[censo]   Variable : HOGAR_NBI_TOT  ("Necesidades básicas insatisfechas")
[censo] -------------------------------------------------------
[censo] Estructura del resultado:
[censo]   Cada fila = una (radio censal x categoria de variable x conteo)
[censo]   Columnas clave: id_geo | codigo_variable | valor_categoria
[censo]                   etiqueta_categoria | conteo
[censo] =======================================================
[censo] Descargando datos desde Hugging Face...


=== comparar — solo NEA ===


[censo] Descarga completa en 0.3s -> 127,347 filas | 13 columnas | 79.3 MB en memoria


,No,Sí,Total
Misiones,91.3,8.7,425667
Corrientes,88.2,11.8,379129
Chaco,88.8,11.2,374487
Formosa,88.3,11.7,198206


In [13]:
# ── Tool: comparar departamentos de una provincia ─────────
print("=== comparar departamentos — Chaco ===")
resultado = comparar(
    "HOGAR_NBI_TOT",
    nivel="departamento",
    provincias=["Chaco"]
)
display(_df(resultado, orient="index").head(10))

[censo] =======================================================
[censo] Consulta al Censo Nacional 2022 (INDEC)
[censo] Fuente: censo-2022-largo.parquet (~137 MB en Hugging Face)
[censo]   (DuckDB descarga solo los bloques que coinciden con los filtros)
[censo] -------------------------------------------------------
[censo]   Variable : HOGAR_NBI_TOT  ("Necesidades básicas insatisfechas")
[censo]   Provincia: Chaco  (codigo INDEC: 22)
[censo] -------------------------------------------------------
[censo] Estructura del resultado:
[censo]   Cada fila = una (radio censal x categoria de variable x conteo)
[censo]   Columnas clave: id_geo | codigo_variable | valor_categoria
[censo]                   etiqueta_categoria | conteo
[censo] =======================================================
[censo] Descargando datos desde Hugging Face...


=== comparar departamentos — Chaco ===


[censo] Descarga completa en 0.2s -> 3,803 filas | 13 columnas | 2.3 MB en memoria


,No,Sí,Total
San Fernando,93.0,7.0,142251
Comandante Fernández,87.3,12.7,33950
General GŁemes,77.7,22.3,22505
Libertador General San Martín,88.4,11.6,21302
Mayor Luis J. Fontana,91.4,8.6,20246
Chacabuco,91.9,8.1,11506
Almirante Brown,84.3,15.7,11475
Quitilipi,85.3,14.7,10421
25 de Mayo,81.2,18.8,9989
9 de Julio,90.9,9.1,9897


In [14]:
# ── Tool: consultar — departamentos cruzados ──────────────
print("=== consultar — dptos específicos de dos provincias ===")
resultado = consultar(
    "HOGAR_NBI_TOT",
    provincias=["Chaco", "Formosa"],
    departamentos=["San Fernando", "Patiño"]
)
display(_df(resultado, orient="index"))

[censo] =======================================================
[censo] Consulta al Censo Nacional 2022 (INDEC)
[censo] Fuente: censo-2022-largo.parquet (~137 MB en Hugging Face)
[censo]   (DuckDB descarga solo los bloques que coinciden con los filtros)
[censo] -------------------------------------------------------
[censo]   Variable : HOGAR_NBI_TOT  ("Necesidades básicas insatisfechas")
[censo]   Provincia: Chaco  (codigo INDEC: 22)
[censo] -------------------------------------------------------
[censo] Estructura del resultado:
[censo]   Cada fila = una (radio censal x categoria de variable x conteo)
[censo]   Columnas clave: id_geo | codigo_variable | valor_categoria
[censo]                   etiqueta_categoria | conteo
[censo] =======================================================
[censo] Descargando datos desde Hugging Face...


=== consultar — dptos específicos de dos provincias ===


[censo] Descarga completa en 0.2s -> 3,803 filas | 13 columnas | 2.3 MB en memoria
[censo] =======================================================
[censo] Consulta al Censo Nacional 2022 (INDEC)
[censo] Fuente: censo-2022-largo.parquet (~137 MB en Hugging Face)
[censo]   (DuckDB descarga solo los bloques que coinciden con los filtros)
[censo] -------------------------------------------------------
[censo]   Variable : HOGAR_NBI_TOT  ("Necesidades básicas insatisfechas")
[censo]   Provincia: Formosa  (codigo INDEC: 34)
[censo] -------------------------------------------------------
[censo] Estructura del resultado:
[censo]   Cada fila = una (radio censal x categoria de variable x conteo)
[censo]   Columnas clave: id_geo | codigo_variable | valor_categoria
[censo]                   etiqueta_categoria | conteo
[censo] =======================================================
[censo] Descargando datos desde Hugging Face...
[censo] Descarga completa en 0.2s -> 1,920 filas | 13 columnas | 1.2 

,No,Sí,Total
San Fernando,93,7,142251


---
## Parte 2 — Test MCP completo (protocolo real)

Levanta el servidor como subproceso y se conecta via el cliente MCP de Python.
Es exactamente el mismo flujo que usa Claude Desktop o Cursor.

Requiere `mcp >= 1.0.0`.

> **Nota Windows:** se pasa `errlog=subprocess.DEVNULL` para evitar un problema de compatibilidad entre el cliente MCP y los streams virtuales del kernel de Jupyter.

In [15]:
from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client

SERVER_PARAMS = StdioServerParameters(
    command=sys.executable,
    args=["-m", "censoargentino.mcp_server"],
)

# En Windows/Jupyter el stderr del kernel no tiene fd real; usamos DEVNULL
ERRLOG = subprocess.DEVNULL

print(f"Servidor: {sys.executable} -m censoargentino.mcp_server")

Servidor: c:\Users\porden\AppData\Local\Programs\Python\Python312\python.exe -m censoargentino.mcp_server


In [16]:
# ── Listar tools y resources disponibles ─────────────────
async def listar_capacidades():
    async with stdio_client(SERVER_PARAMS, errlog=ERRLOG) as (read, write):
        async with ClientSession(read, write) as session:
            await session.initialize()

            tools = await session.list_tools()
            resources = await session.list_resources()

            print("=== Tools disponibles ===")
            for t in tools.tools:
                print(f"  {t.name:25s}  {t.description[:70]}...")

            print()
            print("=== Resources disponibles ===")
            for r in resources.resources:
                print(f"  {str(r.uri):25s}  {r.description[:60]}")

await listar_capacidades()

=== Tools disponibles ===
  buscar_variables           
    Busca variables del censo por nombre o tema.

    Usá esta tool p...
  describir_variable         
    Devuelve los detalles completos de una variable: qué mide, a qué ...
  tabla                      
    Devuelve una tabla con conteo (N) y porcentaje (%) para cada cate...
  comparar                   
    Compara una variable entre unidades geográficas, devolviendo un r...
  consultar                  
    Consulta datos del censo con filtros específicos de provincia y d...

=== Resources disponibles ===
  censo://provincias         
    Tabla de provincias argentinas con código INDEC y nombr
  censo://variables          
    Catálogo completo de variables del censo 2022.
    Cont


In [17]:
# ── Helper: llamar tool via protocolo MCP ────────────────
async def llamar_tool(tool_name: str, args: dict):
    async with stdio_client(SERVER_PARAMS, errlog=ERRLOG) as (read, write):
        async with ClientSession(read, write) as session:
            await session.initialize()
            result = await session.call_tool(tool_name, args)
            return result.content[0].text

# buscar_variables via MCP
print("=== MCP: buscar_variables('internet') ===")
respuesta = await llamar_tool("buscar_variables", {"buscar": "internet"})
for v in json.loads(respuesta):
    print(f"  {v['codigo_variable']:30s}  {v['etiqueta_variable']}")

=== MCP: buscar_variables('internet') ===
  HOGAR_H24A                      El hogar tiene internet en la vivienda
  HOGAR_H24B                      El hogar tiene celular con internet


In [18]:
# ── tabla via MCP ─────────────────────────────────────────
print("=== MCP: tabla('HOGAR_NBI_TOT', provincia='Formosa') ===")
respuesta = await llamar_tool(
    "tabla",
    {"variable": "HOGAR_NBI_TOT", "provincia": "Formosa"}
)
display(_df(respuesta))

=== MCP: tabla('HOGAR_NBI_TOT', provincia='Formosa') ===


,categoria,N,%
0,Sí,23118,11.7
1,No,175088,88.3


In [19]:
# ── comparar via MCP ──────────────────────────────────────
print("=== MCP: comparar NEA por provincia ===")
respuesta = await llamar_tool(
    "comparar",
    {
        "variable": "HOGAR_NBI_TOT",
        "nivel": "provincia",
        "provincias": ["Chaco", "Formosa", "Misiones", "Corrientes"]
    }
)
display(_df(respuesta, orient="index"))

=== MCP: comparar NEA por provincia ===


,No,Sí,Total
Misiones,91.3,8.7,425667
Corrientes,88.2,11.8,379129
Chaco,88.8,11.2,374487
Formosa,88.3,11.7,198206


In [20]:
# ── Leer resource via MCP ─────────────────────────────────
async def leer_resource(uri: str):
    async with stdio_client(SERVER_PARAMS, errlog=ERRLOG) as (read, write):
        async with ClientSession(read, write) as session:
            await session.initialize()
            result = await session.read_resource(uri)
            return result.contents[0].text

print("=== MCP: resource censo://provincias ===")
respuesta = await leer_resource("censo://provincias")
provincias = json.loads(respuesta)
print(f"{len(provincias)} provincias — primeras 5:")
for p in provincias[:5]:
    print(f"  {p['codigo']}  {p['provincia']}")

=== MCP: resource censo://provincias ===
24 provincias — primeras 5:
  02  Ciudad Autónoma De Buenos Aires
  06  Buenos Aires
  10  Catamarca
  14  Córdoba
  18  Corrientes


---
## Resumen

Si todas las celdas corrieron sin errores, el servidor MCP está funcionando correctamente.

**Parte 1** verificó la lógica de cada tool llamándola como Python puro.
**Parte 2** verificó el protocolo MCP completo (stdio → JSON-RPC → respuesta).

El servidor está listo para conectarse con Claude Desktop, Cursor, Cline o cualquier cliente MCP.